In [16]:
import jax
import jax.numpy as jnp
import time

# 模拟输入参数
num_tokens = 10000  # 调大这个值以看到明显的性能差异
half_rotary_dim = 64
mrope_section = [24, 20, 20]
split_indices = [24, 44]

# 模拟数据
cos_all = jax.random.normal(jax.random.PRNGKey(0), (3, num_tokens, half_rotary_dim))
sin_all = jax.random.normal(jax.random.PRNGKey(1), (3, num_tokens, half_rotary_dim))

# --- 方法 1: 你的原始 split 逻辑 ---
@jax.jit
def original_method(cos_all, sin_all):
    cos_splits = jnp.split(cos_all, split_indices, axis=-1)
    sin_splits = jnp.split(sin_all, split_indices, axis=-1)
    
    final_cos_list = [split_tensor[i] for i, split_tensor in enumerate(cos_splits)]
    final_sin_list = [split_tensor[i] for i, split_tensor in enumerate(sin_splits)]
    
    cos = jnp.concatenate(final_cos_list, axis=-1)
    sin = jnp.concatenate(final_sin_list, axis=-1)
    return cos, sin

# --- 方法 2: 优化的 slicing 逻辑 ---
@jax.jit
def optimized_method(cos_all, sin_all):
    indices = [0, 16, 40, 64]
    cos = jnp.concatenate([cos_all[i, :, indices[i]:indices[i+1]] for i in range(3)], axis=-1)
    sin = jnp.concatenate([sin_all[i, :, indices[i]:indices[i+1]] for i in range(3)], axis=-1)
    return cos, sin

def benchmark(name, func, *args):
    # 1. 预热 (Warm-up)
    # 使用 jax.block_until_ready 处理返回的整个元组
    jax.block_until_ready(func(*args))
    
    # 2. 正式测量
    iters = 1000
    start_time = time.perf_counter()
    for _ in range(iters):
        # 同样在这里使用全局函数进行阻塞
        jax.block_until_ready(func(*args))
    end_time = time.perf_counter()
    
    avg_time = (end_time - start_time) / iters * 1000  # 毫秒
    print(f"{name} 平均耗时: {avg_time:.4f} ms")

# 执行测试
print(f"测试规模: num_tokens={num_tokens}\n" + "-"*30)
benchmark("Original (Split)", original_method, cos_all, sin_all)
benchmark("Optimized (Slicing)", optimized_method, cos_all, sin_all)

测试规模: num_tokens=10000
------------------------------
Original (Split) 平均耗时: 0.8851 ms
Optimized (Slicing) 平均耗时: 0.7242 ms


Original Method HLO:
HloModule jit_original_method, is_scheduled=true, entry_computation_layout={(f32[3,10000,64]{2,1,0}, f32[3,10000,64]{2,1,0})->(f32[10000,64]{1,0}, f32[10000,64]{1,0})}, allow_spmd_sharding_propagation_to_parameters={true,true}, allow_spmd_sharding_propagation_to_output={true,true}

%fused_computation (param_0.1: f32[3,10000,64]) -> f32[10000,64] {
  %param_0.1 = f32[3,10000,64]{2,1,0} parameter(0)
  %slice.14 = f32[1,10000,24]{2,1,0} slice(%param_0.1), slice={[0:1], [0:10000], [0:24]}, metadata={op_name="jit(original_method)/slice" source_file="/tmp/ipykernel_1632782/2589552899.py" source_line=21 source_end_line=21 source_column=22 source_end_column=37}
  %bitcast.8 = f32[10000,24]{1,0} bitcast(%slice.14), metadata={op_name="jit(original_method)/slice" source_file="/tmp/ipykernel_1632782/2589552899.py" source_line=21 source_end_line=21 source_column=22 source_end_column=37}
  %slice.13 = f32[1,10000,20]{2,1,0} slice(%param_0.1), slice={[1:2], [0:10000], [24:44]}, m

In [11]:
jax.jit(optimized_method).lower(cos_all, sin_all).compile().as_text()

'HloModule jit_optimized_method, is_scheduled=true, entry_computation_layout={(f32[3,10000,64]{2,1,0}, f32[3,10000,64]{2,1,0})->(f32[10000,64]{1,0}, f32[10000,64]{1,0})}, allow_spmd_sharding_propagation_to_parameters={true,true}, allow_spmd_sharding_propagation_to_output={true,true}\n\n%fused_computation (param_0.1: f32[3,10000,64]) -> f32[10000,64] {\n  %param_0.1 = f32[3,10000,64]{2,1,0} parameter(0)\n  %slice.2 = f32[1,10000,16]{2,1,0} slice(%param_0.1), slice={[0:1], [0:10000], [0:16]}, metadata={op_name="jit(optimized_method)/slice" source_file="/tmp/ipykernel_1632782/2589552899.py" source_line=31 source_end_line=31 source_column=27 source_end_column=65}\n  %bitcast.8 = f32[10000,16]{1,0} bitcast(%slice.2), metadata={op_name="jit(optimized_method)/slice" source_file="/tmp/ipykernel_1632782/2589552899.py" source_line=31 source_end_line=31 source_column=27 source_end_column=65}\n  %slice.1 = f32[1,10000,24]{2,1,0} slice(%param_0.1), slice={[1:2], [0:10000], [16:40]}, metadata={op_na